Propósito: Filtra las empresas del SPY500 para traer los papeles en sobreventa. Usa parametros de los indicadores RSI, MACD y Koncorde en la temporalidad de 60min y 7días.

Este filtro busca capturar movimientos que duran entre 2 a 5 días.

In [1]:
%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [ ]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = df_spy['Symbol'].tolist()

# Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

print(f"Se han cargado {len(tickers_a_analizar)}")
print("tickers_a_analizar:", tickers_a_analizar[:600])

Se han cargado 520
tickers_a_analizar: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK.B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BE', 'BNY', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF.B', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CASY', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CSGP', 'COST', 'CRH', 'CRWD', 'CC

In [ ]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

# Limpiar tickets (Si no existen, usamos una lista pequeña de prueba)
try:
    tickers_limpios = [t.replace('.', '-') for t in tickers_a_analizar]
except NameError:
    print("La lista 'tickers_a_analizar' no existía. Usando lista de prueba...")
    tickers_limpios = ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA', 'META', 'NVDA']



# DESCARGA Y ANÁLISIS.
print(f"Descargando datos para {len(tickers_limpios)} activos...")
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        # kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]
        precio_ant = df['Close'].iloc[-2]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and macd_act <= sig_act 
            #and kon_act < 0
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'Prec_Act': round(precio_act, 2),
                'Prec_Ant': round(precio_ant, 2),
                'RSI_Act': round(rsi_act, 1),
                'RSI_Ant': round(rsi_ant, 1),
                'RSI_MA': round(rsi_ma, 1),
                
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
                
                #'Konk_Act': round(kon_act, 1),
                #'Konk_Ant': round(kon_ant, 1)
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI_Act').to_string(index=False))
else:
    print("Sin señales detectadas.")

Descargando datos para 520 activos...

Ticker  Prec_Act  Prec_Ant  RSI_Act  RSI_Ant  RSI_MA    MACD
  DELL    466.61    465.35     16.1     14.7    22.9 -27.585
  CSCO    109.11    108.81     19.5     15.7    35.2  -1.585
  AVGO    338.34    337.96     19.9     18.1    28.8  -5.514
  SNPS    463.77    463.95     19.9     20.0    26.4  -9.000
  ECHO     83.87     83.81     20.7     20.1    22.5  -2.587
     P    131.26    130.80     20.9     19.2    32.1  -5.309
   PLD    127.74    127.61     21.0     19.2    48.9  -0.591
  CRWD    243.40    243.04     21.0     19.3    21.6  -6.141
   VRT    208.55    208.48     21.3     21.1    26.0  -8.313
  COHR    245.24    244.70     21.7     20.7    28.0 -14.473
   WDC    344.54    343.68     21.7     20.0    28.9 -12.215
   HPE     59.10     58.94     21.8     20.5    21.0  -2.985
  SMCI     34.10     33.98     23.1     21.3    21.1  -1.820
  LITE    847.94    843.61     23.1     20.8    25.2 -49.317
   WMB     69.82     69.78     23.4     22.7  

In [10]:
#1. DESCARGA Y ANÁLISIS DIARIO

print(f"Descargando datos DIARIOS para {len(tickers_limpios)} activos...")
data_1d = yf.download(tickers_limpios, period="1y", interval="1d", group_by='ticker', progress=False)

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()

        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        
        #kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        if (rsi_act <= 35 
            #and kon_act < 0
            ):
        
            empresas_diario.append({
                'Ticker': ticker,
                'Prec_Diario': round(precio_act, 2),
                'RSI_Diario': round(rsi_act, 1),
                'RSI_MA_Diario': round(df['RSI_MA'].iloc[-1], 1),
                #'Konk_S_Act': round(kon_act, 1),
                #'Konk_S_Ant': round(kon_ant, 1),
                
                'MACD_D': round(macd_act, 3),
                'MACD_D_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI_Diario').to_string(index=False))
else:
    print("Sin señales diarias.")

Descargando datos DIARIOS para 520 activos...

PAPELES DETECTADOS EN DIARIO:
Ticker  Prec_Diario  RSI_Diario  RSI_MA_Diario  MACD_D  MACD_D_SIGNAL
   MGM        29.21        17.2           11.7  -2.778         -2.873
   STT       166.46        18.2           30.1  -3.862         -3.146
    MS       179.70        20.3           26.1  -6.241         -5.753
   BAC        51.49        21.5           23.8  -1.873         -1.788
   VRT       209.57        21.9           42.3  -8.236         -5.528
   MOS        18.84        22.4           25.0  -1.350         -1.059
   ALB        97.09        23.8           29.6  -6.088         -5.880
   WDC       345.97        23.9           39.2 -24.300        -16.497
   IVZ        28.16        24.0           35.7  -0.706         -0.471
  SNDK      1361.00        24.8           43.7 -50.920         -4.676
   STX       678.83        24.8           43.9 -36.809        -15.358
   CAT       729.35        25.5           48.5 -14.481         -5.155
   DAL       

In [5]:
# DESCARGA Y ANÁLISIS SEMANAL
print(f"Descargando datos SEMANALES para {len(tickers_limpios)} activos...")
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()

        if len(df) < 50: continue

        # Cálculos Técnicos (La misma lógica, distinta escala temporal)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        
        #kon_act, kon_ant = calcular_koncorde(df)

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO MACRO
        if (rsi_act < 40 and macd_act < sig_act 
            #and kon_act < 0
            ):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'Prec_Seman': round(precio_act, 2),
                'RSI_Sem': round(rsi_act, 1),
                'RSI_MA_S': round(df['RSI_MA'].iloc[-1], 1),
                #'Konk_S_Act': round(kon_act, 1),
                #'Konk_S_Ant': round(kon_ant, 1),
                
                'MACD_S': round(macd_act, 3),
                'MACD_S_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN SEMANAL (VISIÓN MACRO):")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI_Sem').to_string(index=False))
else:
    print("Sin señales semanales.")

Descargando datos SEMANALES para 520 activos...

PAPELES DETECTADOS EN SEMANAL (VISIÓN MACRO):
Ticker  Prec_Seman  RSI_Sem  RSI_MA_S   MACD_S  MACD_S_SIGNAL
   MGM       29.54     18.7      40.7   -2.298         -0.209
  WYNN       76.06     23.8      36.7   -7.886         -6.052
   PNR       53.50     24.6      26.9   -8.633         -8.472
   CRH       82.32     26.2      33.0   -7.555         -6.325
    MS      179.76     27.3      53.2    0.275          5.427
    GS      879.98     27.4      50.5   -6.559         17.556
   URA       35.89     27.4      39.7   -2.345         -1.741
   HWM      217.67     28.3      48.0   -5.097          2.797
   ALB       97.65     28.6      37.2  -12.890         -9.196
  SKYD        7.47     28.8      45.1   -0.553         -0.521
   FRT      105.38     29.0      47.0   -0.654          1.633
   LII      359.20     29.0      32.9  -40.358        -30.846
   MAA      115.94     29.8      42.1   -3.526         -1.544
  CHTR      107.87     30.0      38.3

In [6]:
# FRACTALES ALINEADOS, CRUCE DE 60 MINUTOS AND SEMANAL.

# Convertimos las listas de diccionarios en DataFrames (si no lo eran ya)
df_60 = pd.DataFrame(empresas_60min)
df_sem = pd.DataFrame(empresas_semanal)
print("Alineando las temporalidades.")
if not df_60.empty and not df_sem.empty:
    # Solo quedarán los activos que estén presentes en AMBOS DataFrames
    gangas_alineadas = pd.merge(df_60, df_sem, on='Ticker')


    if not gangas_alineadas.empty:
        print("\n" + "!"*30)
        print("ALINEACIÓN DE TEMPORALIDAD DETECTADA")
        print("Activos en zona de pánico tanto en 60min como en 7d:")
        # Seleccionamos las columnas más relevantes para la comparativa
        columnas_finales = [
            'Ticker', 'Prec_Act', 'RSI_Act', 'Konk_Act',
            'RSI_Sem', 'Konk_S_Act'
        ]
        print(gangas_alineadas[columnas_finales].to_string(index=False))

        print("\nInterpretación: El activo está en suelo macro y dando señal de giro micro.")
    else:
        print("\nNo hay coincidencias 'AND'.")
        print("Existen gangas individuales, pero no están alineadas en ambos fractales hoy.")
else:
    if df_60.empty:
        print("- El fractal de 60 min no detectó señales.")
    if df_sem.empty:
        print("- El fractal Semanal no detectó señales.")
    print("\nNo se puede realizar la operación AND sin datos en ambos marcos.")

Alineando las temporalidades.

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ALINEACIÓN DE TEMPORALIDAD DETECTADA
Activos en zona de pánico tanto en 60min como en 7d:


KeyError: "['Konk_Act', 'Konk_S_Act'] not in index"

In [ ]:
# SEMÁFORO Y REVERSIÓN DE CALIDAD

UMBRAL_PANICO = 35       # Para activos "estirados" al extremo (Cuchillos)
UMBRAL_CALIDAD = 48      # Para captar reversiones inminentes en líderes
EMPRESAS_FILTRADAS = []

print(f"Iniciando escáner sobre {len(tickers_limpios)} activos...")

for ticker in tickers_limpios:
    try:
        # Reutilizamos los datos de la celda de 60 min (Micro Fractal)
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos (Referenciando el fractal de 1h)
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)

        # Variables de control
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        rsi_ma = df['RSI_MA'].iloc[-1]
        kon_act, kon_ant = calcular_koncorde(df)

        # FILTRO ESTIRADO (Filtro de Entrada)
        # Captura el pánico real O sobreventa relativa en empresas fuertes
        if rsi_act < UMBRAL_CALIDAD and kon_act < 0:

            # LÓGICA DEL SEMÁFORO (El Gatillo)
            # Caso Verde: Cruce de Media al alza (Ayer < MA y Hoy > MA)
            if rsi_act > rsi_ma and rsi_ant <= rsi_ma:
                estado = "🟢 GATILLO (Cruce confirmado)"
            # Caso Amarillo: Frenado de caída (Ayer < Hoy pero aún debajo de MA)
            elif rsi_act > rsi_ant:
                estado = "🟡 AGOTAMIENTO (En observación)"
            # Caso Rojo: Inercia bajista (Sigue cayendo)
            else:
                estado = "🔴 CUCHILLO (Sin señal de giro)"

            # Buscamos si el ticker también dio señal en la lista semanal previa
            es_ganga_macro = any(e['Ticker'] == ticker for e in empresas_semanal)
            etiqueta_macro = "🔥 ALINEADO (Macro+Micro)" if es_ganga_macro else "Solo Micro"

            EMPRESAS_FILTRADAS.append({
                'Ticker': ticker,
                'Estado': estado,
                'RSI': round(rsi_act, 1),
                'RSI_MA': round(rsi_ma, 1),
                'Tipo': "Extremo" if rsi_act < UMBRAL_PANICO else "Calidad",
                'Fractal': etiqueta_macro
            })

    except Exception:
        continue

# PRESENTACIÓN DE RESULTADOS
print("\n" + "="*100)
print(f"REPORTE DEL S&P500 - {len(EMPRESAS_FILTRADAS)} ACTIVOS EN RADAR")
print("="*100)

if EMPRESAS_FILTRADAS:
    df_final = pd.DataFrame(EMPRESAS_FILTRADAS)

    # Ordenamos por prioridad: Gatillos primero, luego Agotamiento, luego Cuchillos
    orden_estado = {"🟢 GATILLO (Cruce confirmado)": 0, "🟡 AGOTAMIENTO (En observación)": 1, "🔴 CUCHILLO CAYENDO (Sin señal de giro)": 2}
    df_final['Sort'] = df_final['Estado'].map(orden_estado)
    df_final = df_final.sort_values(by=['Sort', 'RSI']).drop(columns='Sort')

    print(df_final.to_string(index=False))
else:
    print("El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.")

Iniciando escáner sobre 520 activos...

REPORTE DEL S&P500 - 0 ACTIVOS EN RADAR
El mercado no presenta activos por debajo de RSI 48 con pánico de mano débil.
